In [1]:
import spacy
import pandas as pd

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully.")

spaCy model loaded successfully.


In [2]:

text = """
Max Verstappen signed a contract extension with Red Bull Racing in March 2022.
Lewis Hamilton drives for Ferrari in Maranello.
Toto Wolff manages Mercedes AMG F1 from Brackley, United Kingdom.
"""

doc = nlp(text)
print("Text processed into spaCy Doc pipeline.")

Text processed into spaCy Doc pipeline.


In [3]:


entities_data = []

for ent in doc.ents:
    entities_data.append({
        "Entity": ent.text,
        "Label": ent.label_,
        "Description": spacy.explain(ent.label_)
    })


ner_df = pd.DataFrame(entities_data)
print("--- Named Entity Recognition (NER) Results ---")
print(ner_df.to_string(index=False))

--- Named Entity Recognition (NER) Results ---
         Entity  Label                             Description
 Max Verstappen PERSON             People, including fictional
Red Bull Racing    ORG Companies, agencies, institutions, etc.
     March 2022   DATE   Absolute or relative dates or periods
 Lewis Hamilton PERSON             People, including fictional
        Ferrari    ORG Companies, agencies, institutions, etc.
      Maranello    GPE               Countries, cities, states
     Toto Wolff PERSON             People, including fictional
       Mercedes    ORG Companies, agencies, institutions, etc.
       Brackley    GPE               Countries, cities, states
 United Kingdom    GPE               Countries, cities, states


In [4]:
def extract_relation_triples(spacy_doc):
    """
    Extracts Subject-Predicate-Object (SPO) triples from a processed spaCy Doc.
    """
    triples = []

    for sent in spacy_doc.sents:
        subject, action, object_ = "", "", ""

        for token in sent:
            # Find subject (nominal subject)
            if "subj" in token.dep_:
                subject = token.text

            # Find main predicate/verb (ROOT)
            if token.dep_ == "ROOT" or token.pos_ == "VERB":
                action = token.lemma_  # Root form of the verb

            # Find direct object or prepositional object
            if "obj" in token.dep_:
                object_ = token.text

        if subject and action and object_:
            triples.append({
                "Subject": subject,
                "Predicate": action,
                "Object": object_
            })

    return pd.DataFrame(triples)

In [5]:
ie_df = extract_relation_triples(doc)

print("\n--- Information Extraction (SPO Triples) ---")
print(ie_df.to_string(index=False))


--- Information Extraction (SPO Triples) ---
   Subject Predicate    Object
Verstappen      sign     March
  Hamilton     drive Maranello
     Wolff    manage  Brackley
